# Gene Panel Design, Trimming, and Evaluation

This notebook walks through the full pyGeneBasis panel design workflow:

1. **Load data** — `read_adata`
2. **HVG filtering** — `retain_informative_genes`
3. **Panel selection** — `gene_search`
4. **Evaluation** — cell scores, gene scores, cell type mapping
5. **Panel trimming** — `trim_panel` (remove redundant genes)
6. **Visualization**

---

**Prerequisites:**  
- An `.h5ad` file with log-normalised counts in `adata.X` (or a named layer)  
- Cell type labels in `adata.obs` (needed for mapping evaluation)  
- A batch key in `adata.obs` if your data has multiple donors/experiments

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

import pygenebasis as pgb

## 1. Load data

In [ ]:
# ── Edit these paths ──────────────────────────────────────────────────────────
ADATA_PATH    = "/path/to/reference.h5ad"   # log-normalised counts in .X
OUTPUT_DIR    = "/path/to/output"
CELLTYPE_KEY  = "celltype"    # adata.obs column with cell type labels
BATCH_KEY     = "donor_id"    # set to None if single-batch
LAYER         = None          # set to e.g. "lognorm" if counts are in a layer
# ─────────────────────────────────────────────────────────────────────────────

from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

adata = pgb.read_adata(ADATA_PATH)
print(adata)

## 2. HVG filtering

`retain_informative_genes` fits a LOWESS mean-variance trend (scranpy, matching `scran::modelGeneVar`) and keeps genes whose biological variance exceeds the technical noise floor. This reduces the candidate pool from tens of thousands of genes to a few thousand, dramatically speeding up the selection step.

In [ ]:
adata_hvg = pgb.retain_informative_genes(
    adata,
    flavor="scran",    # matches geneBasisR; requires scranpy
    discard_mt=True,   # remove mitochondrial genes
    layer=LAYER,
)
print(f"HVG filtering: {adata.n_vars:,} → {adata_hvg.n_vars:,} genes")
hvg_genes = adata_hvg.var_names.tolist()

In [ ]:
# Save the HVG gene list for later use
pgb.write_csv(
    pd.DataFrame({"gene": hvg_genes}),
    f"{OUTPUT_DIR}/hvg_genes.csv",
    index=False,
)

## 3. Gene panel selection

`gene_search` iteratively adds the gene that most improves neighbourhood preservation in PCA + kNN space. At each step it scores all remaining candidates simultaneously using a vectorised Minkowski distance calculation.

**Key parameters:**
- `n_genes` — target panel size
- `genes_base` — seed genes forced into the panel first (recommended: at least 5)
- `batch_key` / `batch_method` — how to handle multiple donors/batches
- `knn_method="approx"` — use PyNNDescent (fast; use `"exact"` to match R precisely)

In [ ]:
# Optional: seed genes to force into the panel first
# genes_base = ["GENE1", "GENE2", "GENE3"]
genes_base = None

panel_df = pgb.gene_search(
    adata_hvg,
    n_genes=100,
    genes_base=genes_base,
    batch_key=BATCH_KEY,
    knn_method="approx",
    batch_method="per_batch",  # matches geneBasisR default
    n_neighbors=5,
    n_pcs=50,
    layer=LAYER,
    verbose=True,
)

print(panel_df.head(20).to_string(index=False))
panel_genes = panel_df["gene"].tolist()

In [ ]:
# Save the panel
pgb.write_csv(panel_df, f"{OUTPUT_DIR}/gene_panel.csv")
print(f"Panel saved: {len(panel_genes)} genes")

## 4. Evaluation

Three complementary metrics assess panel quality:

| Metric | Function | What it measures |
|---|---|---|
| Cell neighbourhood preservation | `get_neighborhood_preservation_scores` | How well each cell's kNN in the full transcriptome is recovered by the panel |
| Gene prediction score | `get_gene_prediction_scores` | How well each gene's expression is predicted from panel-based kNN neighbours |
| Cell type mapping accuracy | `get_celltype_mapping` | Fraction of cells correctly assigned by kNN majority vote |

In [ ]:
# Pre-compute the true (HVG) graph statistics once — reused by all metrics
from pygenebasis.panel._evaluation import get_neighs_all_stat

neighs_all_stat = get_neighs_all_stat(
    adata_hvg,
    genes_all=hvg_genes,
    batch_key=BATCH_KEY,
    n_neighbors=5,
    n_pcs_all=50,
    knn_method="approx",
    batch_method="per_batch",
    option="approx",  # sample 10% of cells for mean_dist_all — fast for large datasets
    layer=LAYER,
)

### 4a. Cell neighbourhood preservation scores

In [ ]:
cell_scores = pgb.get_neighborhood_preservation_scores(
    adata_hvg,
    genes_selection=panel_genes,
    genes_all=hvg_genes,
    batch_key=BATCH_KEY,
    n_neighbors=5,
    n_pcs_all=50,
    n_pcs_selection=None,  # raw gene space for selection graph
    knn_method="approx",
    batch_method="per_batch",
    neighs_all_stat=neighs_all_stat,
    layer=LAYER,
)

# Attach cell type labels
if CELLTYPE_KEY in adata_hvg.obs.columns:
    cell_scores[CELLTYPE_KEY] = adata_hvg.obs[CELLTYPE_KEY].values

print(f"Cell scores: median={cell_scores['cell_score'].median():.3f}, "
      f"mean={cell_scores['cell_score'].mean():.3f}")
pgb.write_csv(cell_scores, f"{OUTPUT_DIR}/cell_scores.csv")

In [ ]:
# Plot cell score distribution
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(cell_scores["cell_score"].dropna(), bins=80, color="steelblue", edgecolor="none")
ax.axvline(1, color="red", lw=1.2, ls="--", label="score = 1")
ax.set_xlabel("Cell neighbourhood preservation score")
ax.set_ylabel("# cells")
ax.set_title(f"Cell scores — {len(panel_genes)}-gene panel  "
             f"(median={cell_scores['cell_score'].median():.3f})")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Cell scores by cell type (if labels available)
if CELLTYPE_KEY in cell_scores.columns:
    ct_medians = (
        cell_scores.groupby(CELLTYPE_KEY)["cell_score"]
        .median()
        .sort_values()
    )
    fig, ax = plt.subplots(figsize=(6, max(3, len(ct_medians) * 0.28)))
    ax.barh(ct_medians.index, ct_medians.values, color="steelblue")
    ax.axvline(1, color="red", lw=1.2, ls="--")
    ax.set_xlabel("Median cell score")
    ax.set_title("Cell scores by cell type")
    ax.set_xlim(0, max(1.1, ct_medians.max() * 1.05))
    fig.tight_layout()
    plt.show()

### 4b. Gene prediction scores

In [ ]:
gene_scores = pgb.get_gene_prediction_scores(
    adata_hvg,
    genes_selection=panel_genes,
    genes_all=hvg_genes,
    batch_key=BATCH_KEY,
    n_neighbors=5,
    n_pcs_all=50,
    n_pcs_selection=None,
    knn_method="approx",
    batch_method="per_batch",
    layer=LAYER,
)

print(f"Gene scores: median={gene_scores['gene_score'].median():.3f}, "
      f"mean={gene_scores['gene_score'].mean():.3f}")
pgb.write_csv(gene_scores, f"{OUTPUT_DIR}/gene_scores.csv")

In [ ]:
# Bottom-scoring genes (candidates for removal)
gs_sorted = gene_scores.dropna(subset=["gene_score"]).sort_values("gene_score")
n_show = min(40, len(gs_sorted))

fig, ax = plt.subplots(figsize=(5, max(3, n_show * 0.22)))
ax.barh(gs_sorted["gene"].iloc[:n_show], gs_sorted["gene_score"].iloc[:n_show],
        color="salmon")
ax.axvline(1, color="red", lw=1.2, ls="--")
ax.set_xlabel("Gene prediction score")
ax.set_title(f"Bottom {n_show} genes by prediction score")
fig.tight_layout()
plt.show()

### 4c. Cell type mapping accuracy

In [ ]:
if CELLTYPE_KEY in adata_hvg.obs.columns:
    ct_result = pgb.get_celltype_mapping(
        adata_hvg,
        genes_selection=panel_genes,
        celltype_key=CELLTYPE_KEY,
        batch_key=BATCH_KEY,
        n_neighbors=5,
        n_pcs_selection=None,
        knn_method="approx",
        batch_method="harmony",  # default; mnn matches geneBasisR but does not scale
        return_stat=True,
        layer=LAYER,
    )
    mapping_df = ct_result["mapping"]
    ct_stat    = ct_result["stat"]

    # get_celltype_mapping names the truth column "celltype", not CELLTYPE_KEY
    overall_acc = (mapping_df["mapped_celltype"] == mapping_df["celltype"]).mean()
    print(f"Overall mapping accuracy: {overall_acc:.3f}")
    print(ct_stat.sort_values("frac_correctly_mapped").to_string(index=False))

    pgb.write_csv(mapping_df, f"{OUTPUT_DIR}/celltype_mapping.csv")
    pgb.write_csv(ct_stat,    f"{OUTPUT_DIR}/celltype_mapping_stat.csv")

In [ ]:
if CELLTYPE_KEY in adata_hvg.obs.columns:
    # Mapping heatmap
    fig, ax = pgb.plot_mapping_heatmap(
        mapping_df,
        title=f"Cell type mapping — {len(panel_genes)}-gene panel",
    )
    plt.show()

    # Per-celltype accuracy bar chart
    ct_sorted = ct_stat.sort_values("frac_correctly_mapped")
    fig, ax = plt.subplots(figsize=(5, max(3, len(ct_sorted) * 0.3)))
    ax.barh(ct_sorted["celltype"], ct_sorted["frac_correctly_mapped"],
            color="mediumseagreen")
    ax.axvline(1, color="red", lw=1.2, ls="--")
    ax.set_xlabel("Fraction correctly mapped")
    ax.set_title(f"Celltype mapping accuracy  (overall={overall_acc:.3f})")
    ax.set_xlim(0, 1.05)
    fig.tight_layout()
    plt.show()

### 4d. Expression heatmap and co-expression

In [ ]:
if CELLTYPE_KEY in adata_hvg.obs.columns:
    fig, ax = pgb.plot_expression_heatmap(
        adata_hvg, panel_genes,
        celltype_key=CELLTYPE_KEY,
        layer=LAYER,
    )
    plt.show()

In [ ]:
# Co-expression heatmap (first 60 genes for readability)
fig, ax = pgb.plot_coexpression(adata_hvg, panel_genes[:60], layer=LAYER)
ax.set_title("Co-expression — first 60 panel genes")
plt.show()

## 5. Panel trimming

`trim_panel` iteratively removes the most redundant gene from a panel. At each step it uses leave-one-out neighbourhood preservation to score every gene: the gene whose removal costs the least is removed first. This is the inverse of `gene_search` and is useful for reducing a large candidate panel to a smaller target size.

Use `genes_protect` to prevent specific genes from being removed (e.g. known functional markers or experimental controls).

In [ ]:
# Remove 20 genes from the 100-gene panel
n_remove = 20

# Optional: genes that must not be removed
# genes_protect = ["IMPORTANT_GENE1", "IMPORTANT_GENE2"]
genes_protect = None

trim_result = pgb.trim_panel(
    adata_hvg,
    genes_panel=panel_genes,
    n_remove=n_remove,
    genes_all=hvg_genes,
    genes_protect=genes_protect,
    batch_key=BATCH_KEY,
    n_neighbors=5,
    n_pcs_all=50,
    n_pcs_selection=50,
    knn_method="approx",
    batch_method="per_batch",
    layer=LAYER,
    verbose=True,
)

trimmed_panel = trim_result["reduced_panel"]
removal_order = trim_result["removal_order"]
score_traj    = trim_result["score_trajectory"]

print(f"Trimmed panel: {len(panel_genes)} → {len(trimmed_panel)} genes")
print(f"Removed: {removal_order}")

In [ ]:
# Score trajectory during trimming
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(len(score_traj)), score_traj, marker="o", ms=4, color="steelblue")
ax.axvline(0, color="gray", lw=0.8, ls="--", label="full panel")
ax.set_xlabel("Genes removed")
ax.set_ylabel("Mean cell score")
ax.set_title("Score trajectory during trimming")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Save trimmed panel
trimmed_df = pd.DataFrame({"gene": trimmed_panel})
pgb.write_csv(trimmed_df, f"{OUTPUT_DIR}/gene_panel_trimmed.csv", index=False)
print(f"Trimmed panel saved: {len(trimmed_panel)} genes")

## 6. Redundancy analysis

`get_redundancy_stat` performs leave-one-out removal for each panel gene and measures the drop in cell type mapping accuracy. Genes causing a large drop are non-redundant; genes with no impact are candidates for removal.

In [ ]:
if CELLTYPE_KEY in adata_hvg.obs.columns:
    redundancy = pgb.get_redundancy_stat(
        adata_hvg,
        genes=trimmed_panel,
        celltype_key=CELLTYPE_KEY,
        batch_key=BATCH_KEY,
        n_neighbors=5,
        knn_method="approx",
        batch_method="harmony",
        layer=LAYER,
        n_jobs=-1,
    )

    pgb.write_csv(redundancy, f"{OUTPUT_DIR}/redundancy_stat.csv")

    fig, ax = pgb.plot_redundancy_stat(redundancy)
    ax.set_title(f"Gene redundancy — {len(trimmed_panel)}-gene panel")
    plt.show()

---

## CLI equivalent

The same workflow can be run from the command line. See `docs/scripts/run_panel_design.sh` for a complete example.

```bash
# Select a 100-gene panel
pygenebasis panel search \
    --adata reference.h5ad \
    --n-genes 100 \
    --batch-key donor_id \
    --output results/gene_panel.csv

# Trim to 80 genes
pygenebasis panel trim \
    --adata reference.h5ad \
    --panel results/gene_panel.csv \
    --n-remove 20 \
    --batch-key donor_id \
    --output results/gene_panel_trimmed.csv

# Evaluate
pygenebasis panel evaluate \
    --adata reference.h5ad \
    --panel results/gene_panel_trimmed.csv \
    --celltype-key celltype \
    --batch-key donor_id \
    --output results/evaluation.csv
```